# Sector Rotation System — Setup & Backtest

This notebook runs on **Google Colab** (free tier). It:

1. Clones the repo and installs dependencies
2. Populates the database with 2 years of historical data
3. Runs the full pipeline (regime → optimizer → screener → NLP)
4. Executes a walk-forward backtest and displays results
5. Shows the current Executive Summary with dollar allocations

**Runtime:** ~5–8 minutes on Colab free tier.

## 1. Clone Repo & Install Dependencies

In [1]:
# --- Clone private repo using Colab Secrets ---
# Store your GitHub PAT in Colab Secrets (key icon in sidebar) as GITHUB_TOKEN.
# The token needs 'repo' scope for private repository access.

import os
from pathlib import Path

REPO_DIR = '/content/sector-rotation-system'

if not os.path.isdir(REPO_DIR):
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
    !git clone https://{token}@github.com/bigbathtub101/sector-rotation-system.git {REPO_DIR}
else:
    print(f'Repo already cloned at {REPO_DIR}, pulling latest...')
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')

# Install dependencies
!pip install -q -r requirements.txt
print('\nSetup complete.')

Repo already cloned at /content/sector-rotation-system, pulling latest...
remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (9/9), done.
remote: Total 9 (delta 1), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (9/9), 52.91 KiB | 1.76 MiB/s, done.
From https://github.com/bigbathtub101/sector-rotation-system
   1a29e29..3286b70  main       -> origin/main
Updating 1a29e29..3286b70
Fast-forward
 config.yaml                        |   1 +
 data_feeds.py                      |  75 ++++++--
 notebooks/setup_and_backtest.ipynb | 348 +++++++++++++++++++------------------
 portfolio_optimizer.py             |   6 +-
 4 files changed, 248 insertions(+), 182 deletions(-)
Working directory: /content/sector-rotation-system

Setup complete.


## 2. Set API Keys (Optional)

FRED key is optional — the system works without it using placeholder macro data.
Set it here if you have one.

In [2]:
import os
import re

# Optional: set your FRED API key via Colab Secrets (free at https://fred.stlouisfed.org)
# Store your key in Colab Secrets (key icon in sidebar) as FRED_API_KEY.
try:
    from google.colab import userdata
    fred_key = userdata.get("FRED_API_KEY")
    if fred_key:
        os.environ["FRED_API_KEY"] = fred_key
except Exception:
    pass

# Optional: set your SEC EDGAR email via Colab Secrets.
# NOTE: SEC_EDGAR_EMAIL is optional — the email is already in config.yaml.
# Only set this if you want to override the default email in the User-Agent header.
try:
    from google.colab import userdata
    sec_email = userdata.get("SEC_EDGAR_EMAIL")
    if sec_email:
        # Aggressively strip ALL control characters (Colab Secrets injects \r\n)
        sec_email = re.sub(r"[^\x20-\x7E]", "", sec_email).strip()
        if sec_email:
            os.environ["SEC_EDGAR_EMAIL"] = sec_email
        else:
            print("WARNING: SEC_EDGAR_EMAIL contained only control characters — ignoring.")
except Exception:
    pass

# Verify
fred_key = os.environ.get("FRED_API_KEY", "")
print(f"FRED_API_KEY: {('set (' + fred_key[:4] + '...)')  if fred_key else 'NOT SET (will use dummy macro data)'}")
sec_email = os.environ.get("SEC_EDGAR_EMAIL", "")
print(f"SEC_EDGAR_EMAIL: {('set (' + repr(sec_email[:20]) + '...)') if sec_email else 'NOT SET (will use config.yaml default)'}")
# Print repr() to catch any invisible characters that survived
if sec_email:
    print(f"  repr check: {repr(sec_email)}")


FRED_API_KEY: set (a5c3...)
SEC_EDGAR_EMAIL: set ('gabop98@gmail.com'...)
  repr check: 'gabop98@gmail.com'


## 3. Load Config & Backfill Historical Data

Downloads ~2 years of daily prices for all ETFs and watchlist stocks via yfinance,
macro data from FRED (if key is set), and recent SEC filings.

In [3]:
import yaml
import sqlite3
import inspect
import pandas as pd
from pathlib import Path

# Load config
with open('config.yaml', 'r') as f:
    config = yaml.safe_load(f)

# ----- CRITICAL: Define a single DB path used by EVERYTHING -----
DB_PATH = Path(os.getcwd()) / 'rotation_system.db'

# --- Helper: patch a module's DB_PATH everywhere it matters ---
def patch_db_path(mod, new_path):
    """Overwrite a module's DB_PATH global AND fix any function
    defaults that captured the old value at import time."""
    old_path = getattr(mod, 'DB_PATH', None)
    mod.DB_PATH = new_path
    for name, obj in inspect.getmembers(mod, inspect.isfunction):
        if obj.__defaults__:
            new_defaults = []
            changed = False
            for d in obj.__defaults__:
                if isinstance(d, Path) and old_path and d == old_path:
                    new_defaults.append(new_path)
                    changed = True
                else:
                    new_defaults.append(d)
            if changed:
                obj.__defaults__ = tuple(new_defaults)
    print(f'  {mod.__name__}.DB_PATH -> {mod.DB_PATH}')

print("Config loaded. Portfolio total: ${:,.0f}".format(config['portfolio']['total_value']))
print(f"  Taxable: ${config['portfolio']['accounts']['taxable']['value']:,.0f}")
print(f"  Roth IRA: ${config['portfolio']['accounts']['roth_ira']['value']:,.0f}")
print(f"  Sector ETFs: {len(config['tickers']['sector_etfs'])}")
wl_keys = ['watchlist_biotech', 'watchlist_ai_software', 'watchlist_defense', 'watchlist_green_materials']
wl_count = sum(len(config['tickers'].get(k, [])) for k in wl_keys)
print(f"  Watchlist tickers: {wl_count}")
print(f"  DB path: {DB_PATH}")

Config loaded. Portfolio total: $144,000
  Taxable: $100,000
  Roth IRA: $44,000
  Sector ETFs: 11
  Watchlist tickers: 47
  DB path: /content/sector-rotation-system/rotation_system.db


In [4]:
import gc
import sqlite3
import datetime as dt

# Force cleanup of any stale DB connections from prior cell runs
gc.collect()

# Verify DB is accessible before proceeding
if DB_PATH.exists():
    try:
        _test_conn = sqlite3.connect(str(DB_PATH), timeout=30)
        _test_conn.execute("PRAGMA journal_mode=WAL")
        _test_conn.execute("SELECT 1")
        _test_conn.close()
        print(f"✅ DB file verified: {DB_PATH}")
    except sqlite3.OperationalError as e:
        print(f"⚠️ DB appears locked from a prior run, removing stale file: {e}")
        DB_PATH.unlink()

import data_feeds
patch_db_path(data_feeds, DB_PATH)

# Single shared connection — eliminates competing connections that cause DB lock
conn = data_feeds.init_database(DB_PATH)

# Thresholds for incremental ingestion
PRICE_FRESHNESS_DAYS = 3   # skip price download if DB is this fresh
MIN_FILINGS_THRESHOLD = 50  # skip filings download if DB has more than this

# --- START OF PATCH FOR NameError: name 'lookup_cik' is not defined ---
# This patch provides a placeholder `lookup_cik` function to `data_feeds` module
# to prevent a NameError that occurred after a `git pull` updated `data_feeds.py`.
# The actual `lookup_cik` function is missing from `data_feeds.py` or its intended import.
# This temporary fix allows the notebook to proceed but might affect the quality
# of SEC filing data if `lookup_cik` was critical for proper fetching.
# The user should ideally fix `data_feeds.py` directly by defining or importing `lookup_cik`.
if not hasattr(data_feeds, 'lookup_cik'):
    import logging
    logger = logging.getLogger(__name__) # Use a logger to be consistent with data_feeds
    def _placeholder_lookup_cik(ticker, cfg, log_warning=True):
        """
        Placeholder for lookup_cik. Returns None, indicating no CIK found.
        This prevents a NameError and allows SEC fetching to proceed,
        though specific tickers might be skipped if CIK is essential.
        """
        if log_warning:
            logger.warning("Using placeholder lookup_cik for ticker %s. SEC filings for this ticker might be skipped.", ticker)
        return None
    data_feeds.lookup_cik = _placeholder_lookup_cik
    print("⚠️  Applied temporary patch: `data_feeds.lookup_cik` was missing and a placeholder has been added.")
    print("    This allows execution to continue but may affect SEC filing data quality.")
    print("    Consider checking `data_feeds.py` for the correct implementation or import of `lookup_cik`.")
# --- END OF PATCH ---

try:
    # --- Incremental Prices ---
    print("\n📈 Checking price data...")
    latest_date = None
    try:
        row = conn.execute("SELECT MAX(date) FROM prices").fetchone()
        if row and row[0]:
            latest_date = row[0]
    except Exception:
        pass

    today = dt.date.today()
    if latest_date and (today - dt.date.fromisoformat(latest_date)).days <= PRICE_FRESHNESS_DAYS:
        print(f"✅ Prices are current (latest: {latest_date}). Skipping download.")
        prices_summary = {"skipped": True}
    else:
        if latest_date:
            price_start = (dt.date.fromisoformat(latest_date) + dt.timedelta(days=1)).isoformat()
            print(f"📈 Fetching incremental prices from {price_start} to today...")
        else:
            price_start = None
            print("📈 First run — fetching full 2-year price history...")
        prices = data_feeds.fetch_prices(config, start_date=price_start)
        prices, price_warnings = data_feeds.validate_prices(prices, config)
        data_feeds.store_prices(conn, prices)
        prices_summary = {
            "rows": len(prices),
            "tickers": int(prices["ticker"].nunique()) if not prices.empty else 0,
        }
        if price_warnings:
            print(f"  ⚠️ {len(price_warnings)} price warnings")
        print(f"✅ Prices stored: {prices_summary}")

    # --- Macro (always refresh — fast ~5s) ---
    print("\n📊 Fetching macro data from FRED...")
    macro = data_feeds.fetch_macro_data(config)
    macro, macro_warnings = data_feeds.validate_macro(macro)
    data_feeds.store_macro(conn, macro)
    macro_summary = {
        "rows": len(macro),
        "series": int(macro["series_id"].nunique()) if not macro.empty else 0,
    }
    print(f"✅ Macro stored: {macro_summary}")

    # --- SEC Filings (skip if already populated) ---
    print("\n📄 Checking SEC filings...")
    filing_count = conn.execute("SELECT COUNT(*) FROM filings").fetchone()[0]
    if filing_count > MIN_FILINGS_THRESHOLD:
        print(f"✅ Filings already populated ({filing_count:,} rows). Skipping download.")
        print("   To force a refresh, delete the DB file and re-run this cell.")
        filings_summary = {"skipped": True, "existing": filing_count}
    else:
        import time as _time
        max_budget = config.get("sec_edgar", {}).get("max_fetch_time_seconds", 300)
        print(f"📄 First run — fetching SEC filings (budget: {max_budget}s)...")
        _t0 = _time.monotonic()
        filings = data_feeds.fetch_all_filings(config)
        _elapsed = _time.monotonic() - _t0
        data_feeds.store_filings(conn, filings)
        with_text = sum(1 for f in filings if f.get("raw_text"))
        metadata_only = len(filings) - with_text
        filings_summary = {"count": len(filings), "with_text": with_text, "metadata_only": metadata_only}
        print(f"✅ Filings stored: {len(filings):,} total in {_elapsed:.0f}s "
              f"({with_text:,} with text, {metadata_only:,} metadata-only)")
        if len(filings) == 0:
            print("  ⚠️  WARNING: 0 filings were fetched. EFTS may have returned empty results.")
            print("     Check that SEC_EDGAR_EMAIL is correctly set (Colab Secrets or config.yaml).")
            print("     NLP sentiment will run without filing data until this is resolved.")

finally:
    conn.close()

# --- Ticker Coverage Report ---
import pandas as pd
print(f"\n=== Ticker Coverage ===")
print(f"  DB file exists: {DB_PATH.exists()}")
if DB_PATH.exists():
    print(f"  DB file size: {DB_PATH.stat().st_size:,} bytes")
    conn2 = sqlite3.connect(str(DB_PATH))
    try:
        for table in ['prices', 'macro_data', 'filings', 'signals']:
            try:
                count = pd.read_sql(f'SELECT COUNT(*) AS n FROM {table}', conn2).iloc[0]['n']
                print(f"  {table}: {count:,} rows")
            except Exception as e:
                print(f"  {table}: ERROR - {e}")
        loaded = pd.read_sql('SELECT DISTINCT ticker FROM prices', conn2)['ticker'].tolist()
        all_expected = set()
        for key in config['tickers']:
            if isinstance(config['tickers'][key], list):
                all_expected.update(config['tickers'][key])
        missing = all_expected - set(loaded)
        print(f"\n  Tickers loaded: {len(loaded)} / {len(all_expected)} expected")
        if missing:
            print(f"  Missing tickers ({len(missing)}): {sorted(missing)}")
            print(f"  NOTE: OTC ADRs (BAESY, RNMBY, SAABF) and VIX indices")
            print(f"  often fail on yfinance. This is expected and won't")
            print(f"  affect the core ETF-based optimization.")
    finally:
        conn2.close()
else:
    print("  ERROR: DB file was NOT created! Check ingestion logs above.")

✅ DB file verified: /content/sector-rotation-system/rotation_system.db
  data_feeds.DB_PATH -> /content/sector-rotation-system/rotation_system.db

📈 Checking price data...
✅ Prices are current (latest: 2026-02-27). Skipping download.

📊 Fetching macro data from FRED...
✅ Macro stored: {'rows': 610, 'series': 6}

📄 Checking SEC filings...
✅ Filings already populated (224 rows). Skipping download.
   To force a refresh, delete the DB file and re-run this cell.

=== Ticker Coverage ===
  DB file exists: True
  DB file size: 25,677,824 bytes
  prices: 73,000 rows
  macro_data: 610 rows
  filings: 224 rows
  signals: 494 rows

  Tickers loaded: 146 / 152 expected
  Missing tickers (6): ['AAPL', 'AMZN', 'GOOGL', 'META', 'MSFT', 'NVDA']
  NOTE: OTC ADRs (BAESY, RNMBY, SAABF) and VIX indices
  often fail on yfinance. This is expected and won't
  affect the core ETF-based optimization.


## 4. Detect Current Regime

In [5]:
import regime_detector
patch_db_path(regime_detector, DB_PATH)

regime_result = regime_detector.run_regime_detection(config)

print("=" * 60)
print("CURRENT REGIME")
print("=" * 60)
print(f"  Regime:              {regime_result.get('dominant_regime', 'N/A')}")
print(f"  Wedge Volume %:      {regime_result.get('wedge_volume_percentile', 'N/A')}")
print(f"  Fast Shock Risk:     {regime_result.get('fast_shock_risk', 'N/A')}")
print(f"  VIX/RV Ratio:        {regime_result.get('vix_rv_ratio', 'N/A')}")
print(f"  Confirmed:           {regime_result.get('regime_confirmed', 'N/A')}")
print(f"  Consecutive Days:    {regime_result.get('consecutive_days_in_regime', 'N/A')}")

  regime_detector.DB_PATH -> /content/sector-rotation-system/rotation_system.db
CURRENT REGIME
  Regime:              offense
  Wedge Volume %:      84.58
  Fast Shock Risk:     high
  VIX/RV Ratio:        1.5603
  Confirmed:           True
  Consecutive Days:    36


## 5. Run CVaR Optimization

Runs the full CVaR optimizer with Ledoit-Wolf shrinkage and
Longin-Solnik tail correlations. Dollar amounts are split between
Taxable ($100K) and Roth IRA ($44K) using tax-location rules.

In [6]:
import portfolio_optimizer
patch_db_path(portfolio_optimizer, DB_PATH)

regime = regime_result.get('dominant_regime', 'offense')
opt_result = portfolio_optimizer.run_portfolio_optimization(cfg=config, regime=regime)

total = config['portfolio']['total_value']
taxable_val = config['portfolio']['accounts']['taxable']['value']
roth_val = config['portfolio']['accounts']['roth_ira']['value']

print("=" * 80)
print(f"CVaR-OPTIMIZED ALLOCATION (Regime: {regime.upper()})")
print(f"Portfolio: ${total:,.0f} = ${taxable_val:,.0f} Taxable + ${roth_val:,.0f} Roth IRA")
print("=" * 80)

positions = opt_result.get('positions', {})
if positions:
    print(f"\n{'Ticker':<8} {'Weight':>7} {'Total $':>10} {'Taxable $':>10} {'Roth $':>10}  Account")
    print("-" * 75)
    for ticker, info in sorted(positions.items(), key=lambda x: -x[1].get('pct', 0)):
        pct = info.get('pct', 0)
        total_d = info.get('total_dollars', pct / 100.0 * total)
        tax_d = info.get('taxable_dollars', 0)
        roth_d = info.get('roth_dollars', 0)
        acct = info.get('account', 'N/A')
        print(f"{ticker:<8} {pct:>6.1f}% {total_d:>10,.0f} {tax_d:>10,.0f} {roth_d:>10,.0f}  {acct}")

    # Summary by account
    tax_total = sum(info.get('taxable_dollars', 0) for info in positions.values())
    roth_total = sum(info.get('roth_dollars', 0) for info in positions.values())
    print("-" * 75)
    print(f"{'TOTAL':<8} {'100%':>7} {total:>10,.0f} {tax_total:>10,.0f} {roth_total:>10,.0f}")
    print(f"\nAccount utilization: Taxable {tax_total/taxable_val:.0%} | Roth {roth_total/roth_val:.0%}")
else:
    print("No positions returned. Check optimizer logs above.")

  portfolio_optimizer.DB_PATH -> /content/sector-rotation-system/rotation_system.db
CVaR-OPTIMIZED ALLOCATION (Regime: OFFENSE)
Portfolio: $144,000 = $100,000 Taxable + $44,000 Roth IRA

Ticker    Weight    Total $  Taxable $     Roth $  Account
---------------------------------------------------------------------------
VGK        24.8%     35,683     35,683          0  taxable
XLV        10.6%     15,225     15,225          0  taxable
XLC         6.9%      9,910      9,910          0  taxable
XLU         6.0%      8,690      8,690          0  taxable
BIL         5.0%      7,137      7,137          0  taxable
XLY         4.6%      6,619      6,619          0  taxable
XLP         4.0%      5,716      5,716          0  taxable
IONS        4.0%      5,709          0      5,709  roth_ira
NEM         4.0%      5,709          0      5,709  roth_ira
HII         4.0%      5,709          0      5,709  roth_ira
XLI         3.8%      5,441          0      5,441  roth_ira
XLF         3.6%      5,2

## 6. Walk-Forward Backtest

Simulates the system over the historical period, rebalancing monthly.
At each rebalance date, the CVaR optimizer runs with the regime detected
at that point to produce actual per-ETF weights. Returns are computed
using those multi-asset weights, not a simple SPY scaling factor.

In [7]:
import numpy as np
import json as json_module

import portfolio_optimizer
patch_db_path(portfolio_optimizer, DB_PATH)

total = config['portfolio']['total_value']

conn = sqlite3.connect(str(DB_PATH))

# --- Optimization universe (mirrors portfolio_optimizer) ---
opt_universe = (
    config['tickers']['sector_etfs']
    + config['tickers']['geographic_etfs']
    + ['BIL']
)
opt_universe = list(dict.fromkeys(opt_universe))

# Load ETF prices for the optimization universe
placeholders = ','.join(['?'] * len(opt_universe))
etf_prices = pd.read_sql(
    f"SELECT date, ticker, adj_close FROM prices WHERE ticker IN ({placeholders}) ORDER BY date",
    conn, params=opt_universe, parse_dates=['date']
)

# Load SPY as benchmark
spy = pd.read_sql(
    "SELECT date, adj_close FROM prices WHERE ticker='SPY' ORDER BY date",
    conn, parse_dates=['date']
).set_index('date')
spy_returns = spy['adj_close'].pct_change().dropna()

# Load regime signals
signals_df = pd.read_sql(
    "SELECT date, signal_data FROM signals WHERE signal_type='regime_state' ORDER BY date",
    conn, parse_dates=['date']
)
if len(signals_df) > 0:
    signals_df['regime'] = signals_df['signal_data'].apply(
        lambda x: json_module.loads(x).get('dominant_regime', 'offense')
        if isinstance(x, str) else 'offense'
    )
    regime_series = signals_df.set_index('date')['regime']
else:
    regime_series = pd.Series(dtype=str, name='regime')

# Build daily return matrix for ETFs
prices_pivot = etf_prices.pivot(index='date', columns='ticker', values='adj_close')
etf_daily_rets = prices_pivot.pct_change()

# Align to common trading dates with SPY
common_idx = spy_returns.index.intersection(etf_daily_rets.dropna(how='all').index)
spy_returns = spy_returns.loc[common_idx]
etf_daily_rets = etf_daily_rets.loc[common_idx]

print(f"SPY returns: {len(spy_returns)} days")
print(f"ETF universe: {len(opt_universe)} tickers")
print(f"Regime signals: {len(regime_series)} rows")

# --- Identify rebalance dates (first trading day of each month) ---
monthly_first = {}
for date in etf_daily_rets.index:
    key = (date.year, date.month)
    if key not in monthly_first:
        monthly_first[key] = date
rebalance_dates = sorted(monthly_first.values())
print(f"Rebalance dates: {len(rebalance_dates)} months")

# --- Run optimizer at each rebalance date ---
print("Running optimizer for each rebalance date (this may take ~1 min)...")
weights_schedule = {}  # date -> {ticker: weight}
for rb_date in rebalance_dates:
    past = regime_series[regime_series.index <= rb_date]
    regime_at = past.iloc[-1] if len(past) > 0 else 'offense'
    try:
        opt_res = portfolio_optimizer.run_portfolio_optimization(
            conn=conn, cfg=config, regime=regime_at
        )
        positions = opt_res.get('positions', {})
        if positions:
            raw_w = {t: info.get('pct', 0) / 100.0 for t, info in positions.items()}
            available = {t: w for t, w in raw_w.items() if t in etf_daily_rets.columns}
            total_w = sum(available.values())
            if total_w > 0:
                weights_schedule[rb_date] = {t: w / total_w for t, w in available.items()}
    except Exception as e:
        print(f"  ⚠️ Optimizer failed for {rb_date} (regime={regime_at}): {e}")

print(f"Optimizer ran for {len(weights_schedule)}/{len(rebalance_dates)} rebalance dates")

# --- Compute daily portfolio returns using walk-forward weights ---
strategy_rets = []
current_weights = {}
for date in etf_daily_rets.index:
    if date in weights_schedule:
        current_weights = weights_schedule[date]
    if current_weights:
        row = etf_daily_rets.loc[date]
        port_ret = sum(
            w * (row[t] if pd.notna(row.get(t, float('nan'))) else 0.0)
            for t, w in current_weights.items()
        )
    else:
        port_ret = 0.0
    strategy_rets.append(port_ret)

strategy_returns = pd.Series(strategy_rets, index=etf_daily_rets.index, name='strategy_ret')

conn.close()

print(f"Strategy returns computed: {len(strategy_returns)} days")

if len(spy_returns) > 0 and len(strategy_returns) > 0:
    merged = spy_returns.to_frame('spy_ret').join(strategy_returns, how='inner')

    # Attach regime for distribution display
    if len(regime_series) > 0:
        merged['regime'] = regime_series.reindex(merged.index, method='ffill').fillna('offense')
    else:
        merged['regime'] = 'offense'

    merged['spy_cum'] = (1 + merged['spy_ret']).cumprod()
    merged['strategy_cum'] = (1 + merged['strategy_ret']).cumprod()

    days = len(merged)
    ann_factor = 252 / days if days > 0 else 1

    spy_total = merged['spy_cum'].iloc[-1] - 1
    strat_total = merged['strategy_cum'].iloc[-1] - 1

    spy_ann = (1 + spy_total) ** ann_factor - 1
    strat_ann = (1 + strat_total) ** ann_factor - 1

    spy_vol = merged['spy_ret'].std() * np.sqrt(252)
    strat_vol = merged['strategy_ret'].std() * np.sqrt(252)

    spy_sharpe = spy_ann / spy_vol if spy_vol > 0 else 0
    strat_sharpe = strat_ann / strat_vol if strat_vol > 0 else 0

    def max_dd(cum_series):
        peak = cum_series.cummax()
        dd = (cum_series - peak) / peak
        return dd.min()

    spy_mdd = max_dd(merged['spy_cum'])
    strat_mdd = max_dd(merged['strategy_cum'])

    mclean_decay = config['factor_model']['mclean_pontiff_decay']
    alpha_raw = strat_ann - spy_ann
    alpha_adj = alpha_raw * mclean_decay
    mp_label = config['backtest']['mclean_pontiff_label']

    dollar_gain_spy = spy_total * total
    dollar_gain_strat = strat_total * total

    print("=" * 70)
    print("WALK-FORWARD BACKTEST RESULTS")
    print("=" * 70)
    period_start = merged.index[0].strftime('%Y-%m-%d')
    period_end = merged.index[-1].strftime('%Y-%m-%d')
    print(f"Period: {period_start} to {period_end} ({days} trading days)")
    print()
    print(f"{'Metric':<30} {'SPY (B&H)':>15} {'Strategy':>15}")
    print("-" * 60)
    print(f"{'Total Return':<30} {spy_total:>14.1%} {strat_total:>14.1%}")
    print(f"{'Annualized Return':<30} {spy_ann:>14.1%} {strat_ann:>14.1%}")
    print(f"{'Annualized Volatility':<30} {spy_vol:>14.1%} {strat_vol:>14.1%}")
    print(f"{'Sharpe Ratio':<30} {spy_sharpe:>14.2f} {strat_sharpe:>14.2f}")
    print(f"{'Max Drawdown':<30} {spy_mdd:>14.1%} {strat_mdd:>14.1%}")
    print(f"{'Dollar Gain ($144K port.)':<30} {dollar_gain_spy:>13,.0f} {dollar_gain_strat:>13,.0f}")
    print()
    print(f"Raw Alpha vs SPY:      {alpha_raw:>+.2%}")
    print(f"Adjusted Alpha (x{mclean_decay}): {alpha_adj:>+.2%}")
    print(f"  {mp_label}")

    print(f"\nRegime distribution:")
    regime_counts = merged['regime'].value_counts()
    for r, ct in regime_counts.items():
        print(f"  {r}: {ct} days ({ct/days:.0%})")
else:
    print("Not enough data for backtest. Run data ingestion first.")
    print(f"  Debug: SPY returns = {len(spy_returns)}, strategy returns = {len(strategy_returns)}")


  portfolio_optimizer.DB_PATH -> /content/sector-rotation-system/rotation_system.db
SPY returns: 499 days
ETF universe: 22 tickers
Regime signals: 247 rows
Rebalance dates: 24 months
Running optimizer for each rebalance date (this may take ~1 min)...
Optimizer ran for 24/24 rebalance dates
Strategy returns computed: 499 days
WALK-FORWARD BACKTEST RESULTS
Period: 2024-03-04 to 2026-02-27 (499 trading days)

Metric                               SPY (B&H)        Strategy
------------------------------------------------------------
Total Return                            37.1%          35.8%
Annualized Return                       17.3%          16.7%
Annualized Volatility                   16.4%          11.5%
Sharpe Ratio                             1.05           1.46
Max Drawdown                           -18.8%         -11.4%
Dollar Gain ($144K port.)             53,377        51,578

Raw Alpha vs SPY:      -0.54%
Adjusted Alpha (x0.74): -0.40%
  (in-sample, apply 26% McLean-Pontiff d

## 7. Stock Screener — Top Picks

In [8]:
import stock_screener
patch_db_path(stock_screener, DB_PATH)

screener_result = stock_screener.run_stock_screener(cfg=config, regime=regime)

print("=" * 60)
print("THEMATIC WATCHLIST SCORES")
print("=" * 60)

# FIX: The screener returns 'watchlist_scores', not 'watchlists'
watchlists = screener_result.get('watchlist_scores', {})
for name, data in watchlists.items():
    if data:
        print(f"\n--- {name.upper()} ---")
        # data may be a list of dicts (serialized from DataFrame)
        if isinstance(data, list):
            df = pd.DataFrame(data)
        elif hasattr(data, 'to_string'):
            df = data
        else:
            print(f"  (unexpected format: {type(data).__name__})")
            continue
        if not df.empty:
            # Show the most useful columns
            show_cols = [c for c in ['ticker', 'composite_score', 'momentum_score',
                                      'quality_score', 'value_score', 'size_score',
                                      'valuation_label', 'signal'] if c in df.columns]
            if show_cols:
                print(df[show_cols].head(10).to_string(index=False))
            else:
                print(df.head(10).to_string(index=False))
        else:
            print("  (empty)")

# Show ENTRY/EXIT signals grouped by source
signals_out = screener_result.get('signals', {})
if signals_out:
    print(f"\n{'='*60}")
    print("ENTRY / EXIT SIGNALS")
    print(f"{'='*60}")
    for sig_type in ['entry', 'exit']:
        sigs = signals_out.get(sig_type, [])
        if not sigs:
            print(f"\n--- {sig_type.upper()}: none ---")
            continue
        print(f"\n--- {sig_type.upper()} ---")
        # Group by source so both watchlist and sector_screen signals are visible
        watchlist_sigs = [s for s in sigs if s.get('source', 'watchlist') != 'sector_screen']
        sector_sigs = [s for s in sigs if s.get('source') == 'sector_screen']
        if watchlist_sigs:
            print("  [Watchlist]")
            for s in watchlist_sigs[:10]:
                print(f"    {s.get('ticker', '?')} ({s.get('watchlist', '')}): {s.get('reason', '')}")
        if sector_sigs:
            print("  [Sector Screen]")
            for s in sector_sigs[:10]:
                print(f"    {s.get('ticker', '?')} ({s.get('watchlist', '')}): {s.get('reason', '')}")

# --- Sector Top Picks (cross-sector, from overweight sectors) ---
print(f"\n{'='*60}")
print("SECTOR TOP PICKS (best stocks from overweight sectors)")
print(f"{'='*60}")
stp_data = screener_result.get('sector_top_picks', [])
if stp_data:
    stp_df = pd.DataFrame(stp_data)
    show_cols = [c for c in ['ticker', 'etf', 'composite_score', 'momentum_rank',
                               'quality_score', 'value_score', 'valuation_label'] if c in stp_df.columns]
    print(stp_df[show_cols].to_string(index=False))
else:
    print("  (no sector top picks — no overweight sectors identified)")


  stock_screener.DB_PATH -> /content/sector-rotation-system/rotation_system.db
THEMATIC WATCHLIST SCORES

--- BIOTECH ---
ticker  composite_score  quality_score valuation_label
  TERN           3.2959         0.0579 FUNDAMENTAL_BUY
  GPCR           0.7957         0.0415 FUNDAMENTAL_BUY
  IONS           0.7267         0.0765 FUNDAMENTAL_BUY
  HALO           0.6769         0.8842 FUNDAMENTAL_BUY
  EXEL           0.6170         0.8533 FUNDAMENTAL_BUY
  NBIX           0.5496         0.6427 FUNDAMENTAL_BUY
  ALNY           0.5130         0.7192 FUNDAMENTAL_BUY
  BMRN           0.4405         0.5315 FUNDAMENTAL_BUY
  MRNA           0.2007         0.0073 FUNDAMENTAL_BUY
  VKTX           0.1492         0.0281 FUNDAMENTAL_BUY

--- AI_SOFTWARE ---
ticker  composite_score  quality_score  value_score valuation_label
  FSLY           0.5038         0.3928       0.0169           AVOID
  DOCN           0.4414         0.6878       0.1778   MOMENTUM_ONLY
  PLTR           0.3994         0.6280       0.0

## 8. Run Full Monitor (Executive Summary)

In [9]:
# Run the full monitor pipeline and display the executive summary.
# --no-deliver skips Telegram/email/Sheets delivery (not configured in Colab).
# --db explicitly passes our DB path to avoid any path mismatch.
!python monitor.py --no-deliver --db "{DB_PATH}" 2>&1 | head -120

2026-03-01 23:36:08,391 [INFO] monitor: Custom DB path: /content/sector-rotation-system/rotation_system.db — propagating to all modules
2026-03-01 23:36:08,423 [INFO] monitor: Configuration loaded from /content/sector-rotation-system/config.yaml
2026-03-01 23:36:08,424 [INFO] monitor: ════════════════════════════════════════════════════════════
2026-03-01 23:36:08,424 [INFO] monitor:   MONITOR RUN: run_20260301_233608
2026-03-01 23:36:08,425 [INFO] monitor:   Date: 2026-03-01   Mode: LIVE
2026-03-01 23:36:08,425 [INFO] monitor: ════════════════════════════════════════════════════════════
2026-03-01 23:36:08,425 [INFO] monitor: Step 1: Data refresh...
2026-03-01 23:36:08,425 [INFO] data_feeds: Database initialized at /content/sector-rotation-system/rotation_system.db
2026-03-01 23:36:08,426 [INFO] data_feeds: ============================================================
2026-03-01 23:36:08,426 [INFO] data_feeds: STEP 1: Fetching price data
2026-03-01 23:36:08,426 [INFO] data_feeds: =====

## 9. Download Database

Download the populated database to your local machine for use with
the Streamlit dashboard or further analysis.

In [10]:
from google.colab import files

# Download the populated database
files.download(str(DB_PATH))
print("Database downloaded. Place it in the repo root for the dashboard.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Database downloaded. Place it in the repo root for the dashboard.
